# 03 - Custom fine-tuning (LoRA on CatVTON self-attention)

A real, measured experiment - or a clear error message. Steps:

1. check GPU memory, 2. prepare manifests (200 train / 40 val from the official train split, 40 held-out from the official test split),
3. **one-step smoke test** (forward + backward + optimizer step + save), 4. short training with resumable checkpoints,
5. evaluate **baseline vs custom** on the 40 held-out samples, 6. export configuration and metrics.

Important:
* The reference CatVTON repository has **no training script**. This pipeline is our reconstruction from its inference code and
  helper functions; see `docs/research/training-guide.md` for what is and is not from the reference.
* A checkpoint counts as "VirtualFit custom model" only if `metadata.json` says a real optimizer run completed on the real pretrained weights.
* Free runtimes disconnect: re-run the training cell and it resumes from the latest checkpoint in Drive.
* Nothing here tries to stop the runtime from disconnecting or to bypass usage limits; respect the platform's rules.

In [ ]:
# --- Setup: Google Drive (persistent storage) + engine source -------------------------------
# Free runtimes disconnect. Everything that must survive (dataset, checkpoints, outputs) lives in Drive.
from google.colab import drive
drive.mount('/content/drive')

import pathlib, zipfile
WORK = pathlib.Path('/content/drive/MyDrive/virtualfit')
WORK.mkdir(parents=True, exist_ok=True)
ENGINE_ZIP = WORK / 'ai-engine.zip'   # create locally with:  python ai-engine/tools/package_engine.py
assert ENGINE_ZIP.exists(), f'Upload ai-engine.zip to {WORK} first (see ai-engine/README.md)'
if not pathlib.Path('/content/ai-engine').exists():
    zipfile.ZipFile(ENGINE_ZIP).extractall('/content')

%cd /content/ai-engine
!pip -q install -e . --no-deps
!pip -q install "diffusers>=0.31" "transformers>=4.46" "accelerate>=0.31" "peft>=0.13" safetensors lpips scikit-image opencv-python-headless pyyaml pydantic

In [ ]:
# --- GPU memory gate ----------------------------------------------------------------------------
import torch
assert torch.cuda.is_available(), 'No GPU. Change the runtime type (see notebook 01).'
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024**3
print(f'{props.name}: {vram:.1f} GB VRAM')
FORCE_LOW_MEMORY = False
if vram < 12 and not FORCE_LOW_MEMORY:
    raise SystemExit('Less than 12 GB VRAM: training at 512x384 is unlikely to fit. Set FORCE_LOW_MEMORY=True to try anyway '
                     '(consider use_8bit_adam: true and lora_rank: 8 in the config).')

In [ ]:
# --- Dataset location -------------------------------------------------------------------------
# VITON-HD is NOT included in this project and is licensed for research use only.
# Download it yourself from https://github.com/shadow2496/VITON-HD (link in its README) and place the
# unzipped folder at  <WORK>/VITON-HD  so that it contains train/, test/, train_pairs.txt, test_pairs.txt.
VITON_ROOT = WORK / 'VITON-HD'
assert (VITON_ROOT / 'train_pairs.txt').exists(), (
    f'{VITON_ROOT} is missing or incomplete. Download VITON-HD and put it there (see ai-engine/README.md).')
RUN = WORK / 'runs' / 'exp1'
DATA = RUN / 'data'

In [ ]:
# --- Validate the dataset and write train/val/held-out manifests (200 / 40 / 40) ----------------
# train+val come from the official TRAIN split; held-out comes from the official TEST split and is
# used for evaluation only. The command fails loudly if the dataset has too few valid samples.
!python -m virtualfit_engine.dataset.cli prepare --root "{VITON_ROOT}" --out "{DATA}" --n-train 200 --n-val 40 --n-heldout 40

In [ ]:
# --- Write the training config for this Drive layout -----------------------------------------------
import yaml
cfg = yaml.safe_load(open('configs/finetune_catvton_lora.yaml'))
cfg.update(dataset_root=str(VITON_ROOT), train_manifest=str(DATA / 'train.jsonl'),
           val_manifest=str(DATA / 'val.jsonl'), output_dir=str(RUN / 'train'))
if vram < 14:                         # smaller GPUs: be conservative
    cfg.update(use_8bit_adam=True, lora_rank=8, lora_alpha=8)
    !pip -q install bitsandbytes
CFG = RUN / 'finetune.yaml'; CFG.parent.mkdir(parents=True, exist_ok=True)
CFG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(CFG.read_text())

## Step 1 - one-step smoke test

Runs exactly **one optimizer step** into a separate folder (`train_smoke`) and prints the measured peak GPU memory.
Only continue if this succeeds.

In [ ]:
SMOKE = RUN / 'train_smoke'
!python -m virtualfit_engine.training.train --config "{CFG}" --smoke --output-dir "{SMOKE}"
import json
meta = json.loads((SMOKE / 'adapter' / 'metadata.json').read_text())
from virtualfit_engine.training.metadata import is_genuine_finetune
print('optimizer steps completed:', meta['optimizer_steps_completed'])
print('trainable parameters     :', meta['trainable_parameters'], 'of', meta['total_unet_parameters'])
print('real pretrained weights  :', meta['pretrained_weights_loaded'])
print('smoke adapter is genuine :', is_genuine_finetune(meta), '(1 step is a smoke test, not a useful model)')
print(open(SMOKE / 'metrics.jsonl').read())

## Step 2 - training run (resumable)

Default: 300 optimizer steps (~2400 samples seen). If the runtime disconnects, reconnect, re-run the setup cells
and this cell: `--resume latest` continues from the newest complete checkpoint in Drive.

In [ ]:
TRAIN = RUN / 'train'
!python -m virtualfit_engine.training.train --config "{CFG}" --resume latest

In [ ]:
meta = json.loads((TRAIN / 'adapter' / 'metadata.json').read_text())
print('completed:', meta['completed'], '| optimizer steps:', meta['optimizer_steps_completed'])
print('final train loss:', meta['final_train_loss'], '| final val loss:', meta['final_val_loss'])
print('GENUINE custom checkpoint:', is_genuine_finetune(meta))
import matplotlib.pyplot as plt
plt.plot(meta['loss_history']); plt.xlabel('logged step'); plt.ylabel('train loss'); plt.title('Training loss'); plt.show()

## Step 3 - baseline vs custom on the 40 held-out samples

Both columns use the same resident UNet (adapter disabled vs enabled), identical samples, seed, steps and guidance.
FID/KID are intentionally not computed (n=40 is far too small). Fill `manual_artifact_score_1to5` in `results.csv` by eye.

In [ ]:
EVAL = RUN / 'eval'
!python -m virtualfit_engine.evaluation.cli --root "{VITON_ROOT}" --manifest "{DATA / 'heldout.jsonl'}" --adapter "{TRAIN / 'adapter'}" --out "{EVAL}"

In [ ]:
import pandas as pd
from IPython.display import display
df = pd.read_csv(EVAL / 'results.csv')
display(df.groupby('model').mean(numeric_only=True))

In [ ]:
# Visual side-by-side for the first 4 held-out samples
import matplotlib.pyplot as plt
from PIL import Image
ids = df['id'].unique()[:4]
fig, axes = plt.subplots(len(ids), 2, figsize=(7, 5 * len(ids)))
for r, i in enumerate(ids):
    for c, m in enumerate(('baseline', 'custom')):
        axes[r][c].imshow(Image.open(EVAL / 'images' / f'{i}__{m}.png')); axes[r][c].set_title(f'{i} - {m}'); axes[r][c].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# --- Export configuration and metrics for the experiment log -----------------------------------------
import shutil
EXPORT = WORK / 'exports' / 'exp1'; EXPORT.mkdir(parents=True, exist_ok=True)
for src in (CFG, TRAIN / 'config.json', TRAIN / 'metrics.jsonl', TRAIN / 'adapter' / 'metadata.json',
            EVAL / 'results.csv', EVAL / 'summary.json', EVAL / 'report.md', DATA / 'summary.json'):
    shutil.copy(src, EXPORT / src.name if src.name != 'summary.json' else EXPORT / f'{src.parent.name}_summary.json')
print('exported to', EXPORT, '- copy these into docs/research/experiment-log.md; do NOT commit the adapter if it was trained on data you cannot share.')